LOADING THE LIBRARIES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!pip install groq

MODULE 0
Loading Groq

In [ ]:
import os
from groq import Groq

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [ ]:
client = Groq()

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Explain why Groq's inference is fast in 3 sentences."},
    ],
)

print(response.choices[0].message.content)

LOADING THE COURSE DATASET

In [ ]:
course=pd.read_csv('/content/coursera_course_dataset_v3.csv')

PRINTING THE FIRST 5 ROWS

In [ ]:
course.head()

We drop the unnamed column as it is of no use

In [ ]:
course.rename(columns={'Unnamed: 0':'Course_id'},inplace=True)

In [ ]:
course.head()

In [ ]:
course.columns

In [ ]:
course.columns=course.columns.str.lower()


In [ ]:
course.head()

In [ ]:
#print rows and columns
course.shape

The data has 623 rows and 11 columns

In [ ]:
role_skills={'Data Scientist': ['python', 'statistics', 'machine learning', 'sql',
                           'data visualization', 'deep learning'], 'Data Analyst': ['excel', 'sql', 'statistics', 'data visualization', 'python'], 'ML Engineer':['python', 'machine learning', 'deep learning', 'sql', 'cloud'],'Data Engineer': ['python','sql', 'data wrangling', 'cloud', 'big data', 'apis'], 'Business Analyst': ['excel', 'sql', 'statistics', 'data visualization',
                           'communication'],'BI Developer': ['sql', 'data visualization', 'excel', 'statistics', 'power bi'],'AI Researcher':['python', 'deep learning', 'machine learning',
                           'mathematics', 'nlp'], 'Backend Developer': ['python', 'sql', 'apis', 'cloud', 'git'],'MLOps Engineer':['python', 'machine learning', 'cloud', 'docker', 'mlops'], 'NLP Engineer':['python', 'machine learning', 'deep learning', 'nlp',
                           'statistics']



}

In [ ]:
role_skills['Data Scientist']

In [ ]:
course.isnull().sum()

In [ ]:
course.duplicated().sum()

No duplicates as such

In [ ]:
course.drop(columns=['organization','ratings','course_url' ,	'course_students_enrolled' ,'course_description', 'review count','difficulty','type','duration'],axis=1,inplace=True)

In [ ]:
course.isnull().sum()

In [ ]:
course.head()

In [ ]:
course.shape

In [ ]:
#cleaning the skills dataset
import re
def clean(text):
  return re.sub(r'[^a-z0-9]','',text.str.lower())
  return re.sub(r'\s+',"",text).strip()

In [ ]:
import re

def clean(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()

# build the search text from Title + Skills  (note the " " between them)
course['text'] = (course['title'] + " " + course['skills']).apply(clean)

# keep a clean LIST of skills — M6 needs this
course['skills'] = course['skills'].fillna("").apply(
    lambda s: [x.strip().lower() for x in str(s).split(",") if x.strip()])

course[['course_id', 'title', 'skills', 'text']].head(3)

In [ ]:
course.head()

In [ ]:
course['text'].iloc[0]

In [ ]:
target_role = input('Enter a role: ').strip()
assert target_role in role_skills, f"'{target_role}' is not in the role list. Pick one of: {list(role_skills)}"

In [ ]:
target_role

In [ ]:
current_skills = [s.strip().lower() for s in input('Enter your current skills (comma-separated): ').split(",") if s.strip()]
have = set(current_skills)


In [ ]:
have

In [ ]:
skill_gap = [s for s in role_skills[target_role] if s.lower() not in have]
query_text = " ".join(skill_gap)
print("skill_gap:", skill_gap)

In [ ]:
query_text

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf=TfidfVectorizer(stop_words='english')

In [ ]:
course_vector=tfidf.fit_transform(course['text']).toarray()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def recommend(query_text):
  query_vector=tfidf.transform([query_text])
  similarity=cosine_similarity(course_vector,query_vector).flatten()
  course_list=course.copy()
  course_list['score']=similarity
  course_list=course_list.sort_values(by='score',ascending=False)
  return course_list.head(10)

In [ ]:
result_tfdif=recommend(query_text)

In [ ]:
result_tfdif

In [ ]:
from sentence_transformers import SentenceTransformer,util

In [ ]:
model=SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
course_embedding=model.encode(course['text'].tolist(),convert_to_tensor=False)

In [ ]:
course_embedding

In [ ]:
def recommend2(query_text):
  query_vector2=model.encode([query_text],convert_to_tensor=False)
  similarity=cosine_similarity(course_embedding,query_vector2).flatten()
  course_list=course.copy()
  course_list['score_embedding']=similarity
  course_list=course_list.sort_values(by='score_embedding',ascending=False)
  return course_list.head(10)

In [ ]:
result_embedding=recommend2(query_text)

In [ ]:
result_embedding

In [ ]:
def is_relevant(course_skills,skill_gap):
  course_skills=set(course_skills)
  skill_gap=set(skill_gap)
  return len(course_skills & skill_gap)>0

In [ ]:
def precision_at_k(recommendations,skill_gap,k):
  relevant=0
  for _,row in recommendations.head(10).iterrows():
    if is_relevant(row['skills'],skill_gap):
      relevant+=1

  return relevant/k

In [ ]:
skill_gap

In [ ]:
precisiontfdif=precision_at_k(result_tfdif,skill_gap,10)
print(precisiontfdif)

In [ ]:
precision_embedding=precision_at_k(result_embedding,skill_gap,10)
print(precision_embedding)

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Method": ["TF-IDF","Embedding"],
    "Precision@10": [
        precisiontfdif ,
        precision_embedding
    ]
})

comparison

In [ ]:
import matplotlib.pyplot as plt

comparison.plot(
    x="Method",
    y="Precision@10",
    kind="bar",
    legend=False
)

plt.ylabel("Precision@10")
plt.title("TF-IDF vs Embedding")
plt.show()